# Day 2 概念实验：多头注意力与位置编码

本 notebook 用可运行的小实验回答三个问题：**多个头是否会产生不同的注意力分布？拼接后维度如何保持不变？没有位置编码时，注意力为何没有词序概念？**

公式推导与延伸阅读见同目录 `ima/第1周-Day2-多头注意力与位置编码.md`；这里验证核心结论，不搬运阅读材料。

## 实验 1：两个头是否真的有不同视角？

手写 `softmax(QK^T / sqrt(d_head))V`。两个头使用独立投影矩阵，检查权重行和为 1、拼接后维度恢复为 `d_model`。

In [ ]:
import numpy as np
rng = np.random.default_rng(7)
tokens = ["小明", "把", "书", "给了"]
seq_len, d_model, num_heads = 4, 8, 2
head_dim = d_model // num_heads
X = rng.normal(size=(seq_len, d_model))
Wq = rng.normal(scale=0.45, size=(num_heads, d_model, head_dim))
Wk = rng.normal(scale=0.45, size=(num_heads, d_model, head_dim))
Wv = rng.normal(scale=0.45, size=(num_heads, d_model, head_dim))
def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)
weights, outputs = [], []
for h in range(num_heads):
    Q, K, V = X @ Wq[h], X @ Wk[h], X @ Wv[h]
    a = softmax(Q @ K.T / np.sqrt(head_dim))
    weights.append(a); outputs.append(a @ V)
head_weights = np.stack(weights)
concatenated = np.concatenate(outputs, axis=-1)
assert np.allclose(head_weights.sum(axis=-1), 1.0)
assert concatenated.shape == (seq_len, d_model)
print("head_dim =", head_dim, "; 拼接输出 shape =", concatenated.shape)
print("头1 对‘小明’:", dict(zip(tokens, np.round(head_weights[0, 0], 3))))
print("头2 对‘小明’:", dict(zip(tokens, np.round(head_weights[1, 0], 3))))
print("两头权重平均绝对差 =", round(float(np.abs(head_weights[0]-head_weights[1]).mean()), 3))

## 实验 2：把两个头的权重并排看

行是查询 token，列是被关注 token。不同投影会产生不同分布；这说明多头提供并行的表示子空间，但不意味着每个头一定对应一个人类可命名的语法规则。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
fig, axes = plt.subplots(1, num_heads, figsize=(9, 3.6), sharey=True)
for h, ax in enumerate(axes):
    im = ax.imshow(head_weights[h], vmin=0, vmax=1, cmap="YlOrRd")
    ax.set_title(f"头 {h+1} 的注意力")
    ax.set_xticks(range(seq_len), tokens); ax.set_yticks(range(seq_len), tokens)
    ax.set_xlabel("被关注 token")
    if h == 0: ax.set_ylabel("查询 token")
    for i in range(seq_len):
        for j in range(seq_len): ax.text(j, i, f"{head_weights[h,i,j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=axes, shrink=0.8, label="注意力权重")
fig.suptitle("同一输入的多头注意力")
plt.tight_layout(); plt.show()

## 实验 3：正弦/余弦编码如何让位置变得可区分？

生成多个位置的编码，验证位置向量互不相同；把同一个 token 放在两个位置，观察加法注入位置后输入发生变化。

In [ ]:
import numpy as np
def sinusoidal_encoding(length, width):
    pos = np.arange(length)[:, None]
    dims = np.arange(0, width, 2)[None, :]
    angles = pos / (10000 ** (dims / width))
    pe = np.zeros((length, width))
    pe[:, 0::2], pe[:, 1::2] = np.sin(angles), np.cos(angles)
    return pe
PE = sinusoidal_encoding(32, 16)
unit = np.ones(16)
assert len(np.unique(PE, axis=0)) == len(PE)
print("位置0与位置1余弦相似度 =", round(float(PE[0]@PE[1]/np.linalg.norm(PE[0])/np.linalg.norm(PE[1])), 3))
print("位置0与位置16余弦相似度 =", round(float(PE[0]@PE[16]/np.linalg.norm(PE[0])/np.linalg.norm(PE[16])), 3))
print("同一 token 在位置0/7 的编码输入差异范数 =", round(float(np.linalg.norm((unit+PE[0])-(unit+PE[7]))), 3))

## 实验 4：不同频率共同组成位置“指纹”

热力图中左侧维度变化快、右侧维度变化慢。多种频率叠加使每个位置得到可区分的编码。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(PE, aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
ax.set_title("正弦位置编码：位置 × 维度"); ax.set_xlabel("嵌入维度"); ax.set_ylabel("位置")
fig.colorbar(im, ax=ax, label="编码值")
plt.tight_layout(); plt.show()